# YuE2 Kaggle Notebook Demo

This demo shows how to run **YuE2** on a **Kaggle Notebook (NVIDIA Tesla T4 single or dual 16 GB VRAM)**.

YuE2 includes **automatic hardware detection & multi-GPU distribution**:
- **Multi-GPU (Dual T4)**: Automatically distributes the `YuE2-3B` transformer layers across `cuda:0` and `cuda:1` via `device_map="auto"` and places the VAE decoder on `cuda:1`.
- **Single GPU (T4/P100 $\le$ 18 GB VRAM)**: Automatically sets `offload_ar = True` during non-autoregressive synthesis to conserve ~4 GB VRAM, adjusts `memory_budget_gib` to $\le$ 13.0 GiB, and limits `query_chunk_size = 512` in attention to prevent temporary memory spikes on long songs.
- Uses tiled VAE decoding (`vae_core_frames = 512`).
- Gracefully falls back for BF16 and FlashAttention without crashing.

In [ ]:
# Step 1: Install YuE2 package in editable mode
!pip install -e .

In [ ]:
# Step 2: Check GPU environment
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"- GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GiB")

In [ ]:
# Step 3: Load YuE2 Pipeline
# Note: No manual memory flags required — YuE2 auto-adapts to single or dual T4!
from yue2 import YuE2Pipeline

pipe = YuE2Pipeline.from_pretrained("m-a-p/YuE2-3B", device="cuda")

print("\n--- Auto-Configured Settings ---")
print(f"Memory Budget:    {pipe.memory_budget_gib} GiB")
print(f"AR Offloading:    {pipe.offload_ar}")
print(f"Multi-GPU Mode:   {pipe.multi_gpu}")
print(f"VAE Device:       {pipe.vae_device}")
print(f"VAE Core Frames:  {pipe.vae_core_frames}")
print(f"Main Device:      {pipe.device}")

In [ ]:
# Step 4: Generate a Song
request = {
    "style": "acoustic pop, warm guitar, gentle piano, soothing female vocal",
    "lyrics": "[Verse]\nWalking through the morning light\nEverything will be alright\n[Chorus]\nA gentle melody in the air\nBringing peace and love to share",
    "cot": "full",
    "seed": 42
}

song = pipe(**request)

# Save generated audio & metadata artifacts
output_dir = "outputs/kaggle_demo"
song.save_artifacts(output_dir)
print(f"Successfully saved song artifacts to {output_dir}")

In [ ]:
# Step 5: Play Generated Audio in Notebook
from IPython.display import Audio

Audio(f"{output_dir}/audio.flac")